In [27]:
import os
os.chdir("/home/jovyan/work/MST")  # adjust if needed
os.getcwd()



'/home/jovyan/work/MST'

In [66]:
import torch
from pathlib import Path

from mst.models.dino import DinoV2ClassifierSlice
from mst_xai.xai_methods.gradcam_slice_level import GradCAM_Slice
from mst_xai.xai_methods.gradcam_patch_level import GradCAM_MST
from mst.data.datasets.dataset_3d_odelia import ODELIA_Dataset3D

In [67]:
ckpt_dir = Path("runs/ODELIA/DinoV2ClassifierSlice_Final")
model = DinoV2ClassifierSlice.load_best_checkpoint(ckpt_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


Using cache found in /home/jovyan/.cache/torch/hub/facebookresearch_dinov2_main
/opt/conda/lib/python3.11/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer was not TransformerEncoderLayer
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


DinoV2ClassifierSlice(
  (loss_func): CrossEntropyLoss()
  (auc_roc): ModuleDict(
    (train_): MulticlassAUROC()
    (val_): MulticlassAUROC()
    (test_): MulticlassAUROC()
  )
  (acc): ModuleDict(
    (train_): MulticlassAccuracy()
    (val_): MulticlassAccuracy()
    (test_): MulticlassAccuracy()
  )
  (encoder): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
      (norm): Identity()
    )
    (blocks): ModuleList(
      (0-11): 12 x NestedTensorBlock(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): MemEffAttention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
   

In [68]:
ds = ODELIA_Dataset3D(split="test")

sample = ds[0]   # single sample
uid = sample["uid"]
gt = int(sample["target"])

print("UID:", uid, "GT:", gt)


UID: ODELIA_BRAID1_0246_1_left GT: 0


In [69]:
batch = {
    "source": sample["source"].unsqueeze(0).to(device),
    "target": torch.tensor([gt], device=device)
}


In [70]:
batch["source"].shape
# expected: [1, 3, D, H, W]


torch.Size([1, 1, 32, 224, 224])

In [71]:
logits = model(batch["source"])
pred = logits.argmax(dim=1).item()

print("Predicted class:", pred)


Predicted class: 0


In [72]:
gradcam = GradCAM_MST(model)


In [73]:
sal = gradcam.generate(batch, target_class=pred)


RuntimeError: The size of tensor a (32) must match the size of tensor b (33) at non-singleton dimension 1

In [ ]:
# Inside GradCAM_Slice, but we test externally by re-running backward
model

logits = model(batch["source"])
score = logits[:, pred].sum()
score.backward()

acts = gradcam.activations          # [B, N, C]
grads = gradcam.gradients           # [B, N, C]

print("CLS grad sum:", grads[:, 0, :].abs().sum().item())
print("Slice grad sum:", grads[:, 1:, :].abs().sum().item())


CLS grad sum: 9.6312894821167
Slice grad sum: 0.0


# Summary
The classifier depends ONLY on the CLS token.
Slice tokens receive ZERO gradient.

In [ ]:
print("CAM shape:", sal.shape)
print("min / max / std:", sal.min(), sal.max(), sal.std())


CAM shape: torch.Size([32])
min / max / std: tensor(0., device='cuda:0', grad_fn=<MinBackward1>) tensor(0., device='cuda:0', grad_fn=<MaxBackward1>) tensor(0., device='cuda:0', grad_fn=<StdBackward0>)
